# Lentils foreign-object segmentation with DynUNet — inference

Load a pipeline trained by [`01_train.ipynb`](01_train.ipynb), run it over the **180-frame test
split** with a `Predictor`, and report:

- **segmentation metrics** — foreground IoU / Dice / pixel accuracy — read straight off the
  pipeline's `SegMetrics` node (the node-metric idiom: the node accumulates during the pass, you read
  `compute()`), and
- **qualitative panels** — false-colour scene, foreground-probability heatmap, and prediction-vs-GT
  contours for a few object frames.

At inference the graph runs `Norm → DynUNet` (the losses are stage-gated off and the augmentation is
an identity passthrough); DynUNet tiles each full frame with Gaussian-blended overlaps, `tile_batch`
packing the tiles onto the batch axis (~16× faster, output-identical).

> **Prerequisites**
>
> 1. Install the plugin with the notebooks extra: `uv sync --extra notebooks` from the repo root,
>    plus `cuvis-ai-augment` (the trained graph contains its augmentation node).
> 2. From the repo root, launch with `uv run jupyter lab`.
> 3. Train + save a pipeline first (run `01_train.ipynb`). This notebook loads it from `PIPELINE_DIR`
>    (configuration cell; defaults to the train notebook's `outputs/lentils_unet_run/trained_models`).
>
> **Data**: reuses the per-frame NPZ the train notebook wrote to `outputs/npz_local`
> (`universe.csv` + `splits.json`); if absent it downloads + converts with
> `PublicDatasets.download_dataset` + `convert_split_manifest` (needs `cuvis-ai-dataloader[cu3s,coco]`
> + the Cuvis SDK). The test split is resolved from the splits.json.

In [ ]:
# Colab bootstrap: no-op when running locally
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("In Colab Env")
    %pip install -q cuvis-ai cuvis-ai-unet cuvis-ai-augment "cuvis-ai-dataloader[cu3s,coco]"

    import torch

    if not torch.cuda.is_available():
        print(
            "WARNING: No GPU detected. Switch via Runtime > Change runtime type > GPU. "
            "DynUNet inference on CPU is slow."
        )

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from cuvis_ai_core.data.public_datasets import PublicDatasets
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.training import Predictor
from cuvis_ai_core.utils.node_registry import NodeRegistry
from cuvis_ai_dataloader.data import MultiNpzDataModule
from cuvis_ai_dataloader.data.npz_converter import convert_split_manifest
from cuvis_ai_schemas.enums import ExecutionStage
from cuvis_ai_schemas.training import DataSplitConfig
from loguru import logger

import utils

In [ ]:
# Big frames (~263 MB) + DataLoader workers can exhaust /dev/shm under the default sharing strategy;
# file_system avoids it (harmless here at num_workers=0, matches the training notebook).
torch.multiprocessing.set_sharing_strategy("file_system")

logger.remove()
logger.add(sys.stderr, level="INFO")

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
logger.info(f"Using device {device}")

## Tutorial configuration

- **`PIPELINE_DIR`**: a train notebook's `trained_models` dir (holds the single `*.yaml` + its `.pt`).
  Defaults to the trainrun output; the manual path writes `lentils_unet_manual.*` in the same dir.
- **`TEST_LIMIT`**: 0 evaluates the full test split; N evaluates the first N frames.
- **`PANEL_SCAN`**: how many leading test frames to scan for object-frame panels (section 4).
- **`SMOKE_LIMIT`**: only used if the NPZ isn't already on disk — frames/split to convert here (keep
  it equal to what `01_train.ipynb` used so the test split matches).

In [ ]:
PIPELINE_DIR = Path("outputs/lentils_unet_run/trained_models")  # a train notebook's output dir
TEST_LIMIT = 0     # 0 = full 180-frame test split; N = first N frames
PANEL_SCAN = 24    # leading test frames to scan for object-frame panels
SMOKE_LIMIT = 12   # frames/split if converting fresh here (match 01_train.ipynb; ignored if NPZ exists)

PIPE_YAML, PIPE_PT = utils.resolve_pipeline(PIPELINE_DIR)
print(f"Pipeline dir:  {PIPELINE_DIR}")
print(f"Pipeline YAML: {PIPE_YAML}")
print(f"Weights:       {PIPE_PT} ({PIPE_PT.stat().st_size / 1e6:.1f} MB)")
print(f"Test limit:    {TEST_LIMIT or 'full (180)'}")

## 1 · Load the trained pipeline + data

Register the unet + augment plugins (so the saved node classes resolve), then
`CuvisPipeline.load_pipeline` rebuilds the graph and loads the weights. The NPZ (reused from the
train notebook, or downloaded + converted here) is served by `MultiNpzDataModule`; `setup("predict")`
prepares the predict split (a copy of `test`).

In [ ]:
registry = NodeRegistry()
registry.register_plugin(str(utils.UNET_MANIFEST))
registry.register_plugin(str(utils.AUGMENT_MANIFEST))
pipeline = CuvisPipeline.load_pipeline(
    str(PIPE_YAML), weights_path=str(PIPE_PT), device=str(device), node_registry=registry
)
pipeline.torch_layers.eval()
print("Loaded:", pipeline.name, "| nodes:", [n.name for n in pipeline.nodes])

# Fetch + convert the dataset to per-frame NPZ (skipped when the artifacts already exist -- the
# train notebook writes them to the same outputs/npz_local).
LENTILS_DATASET_NAME = "industrial_fod_lentils"
NPZ_DIR = Path("outputs/npz_local")
SPLITS_JSON = NPZ_DIR / "splits.json"
UNIVERSE_CSV = NPZ_DIR / "universe.csv"
if not (SPLITS_JSON.is_file() and UNIVERSE_CSV.is_file()):
    dataset_dir = Path("/content/data") if IN_COLAB else Path("../../data")
    raw_dir = dataset_dir / "XMR_Industrial_Foreign_Object_Detection_Lentils"
    PublicDatasets.download_dataset(
        LENTILS_DATASET_NAME, download_path=str(dataset_dir), force=False
    )
    result = convert_split_manifest(
        raw_dir / "splits.csv", raw_dir, NPZ_DIR,
        universe_csv=UNIVERSE_CSV, splits_json=SPLITS_JSON,
        limit=SMOKE_LIMIT,  # 0 = convert all frames; must match what 01_train.ipynb used
    )
    SPLITS_JSON, UNIVERSE_CSV = result.splits_json, result.universe_csv

datamodule = MultiNpzDataModule(
    splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
    universe_csv=str(UNIVERSE_CSV),
    batch_size=1,
    num_workers=0,
)
datamodule.setup(stage="predict")  # Predictor evaluates the predict split (a copy of test)
print("Predict (= test) frames:", len(datamodule.predict_ds))

### Pipeline graph

Inline view of the restored graph. A `CuvisPipeline` renders itself in Jupyter as an SVG built from
Graphviz DOT (needs the system `dot` binary; it falls back to a Mermaid source block otherwise).

In [ ]:
pipeline

## 2 · Run inference on the test split

`Predictor` runs the loaded pipeline over the split — no hand-rolled forward loop. With
`stage=ExecutionStage.TEST` the pipeline's `SegMetrics` node fires and accumulates per-frame
foreground IoU / Dice + pixel accuracy across this single pass. `collect_outputs=False` keeps memory
flat (the metrics live on the node); section 4 does a second, bounded pass that keeps the logits for
the panels.

In [ ]:
Predictor(pipeline, datamodule).predict(
    stage=ExecutionStage.TEST,
    collect_outputs=False,
    max_batches=(TEST_LIMIT or None),
)
print("Test pass complete.")

## 3 · Segmentation metrics

Read straight off the `SegMetrics` node, which accumulated during the pass above: foreground IoU /
Dice are averaged over object frames (frames with no foreground are skipped, matching the champion
evaluation), pixel accuracy is over all evaluated pixels. A 1-epoch tutorial model sits far below the
champion reference — the comparison line shows the gap.

In [ ]:
seg = {n.name: n for n in pipeline.nodes}["SegMetrics"]
m = seg.compute()
champ = utils.CHAMPION["2d_128"]
print(f"fg_iou    = {m['fg_iou']:.4f}   (champion {champ['fg_iou']:.4f})")
print(f"fg_dice   = {m['fg_dice']:.4f}   (champion {champ['fg_dice']:.4f})")
print(f"pixel_acc = {m['pixel_acc']:.4f}")

## 4 · Qualitative panels

A second, bounded `Predictor` pass keeps DynUNet's per-frame `logits`; for the first few object
frames it softmaxes them into a foreground-probability map and argmaxes into a predicted mask. Cubes
are reloaded from their NPZ for a true false-colour view (red = GT contour, cyan = prediction).

In [ ]:
collected = Predictor(pipeline, datamodule).predict(
    stage=ExecutionStage.TEST,
    collect_outputs=True,
    max_batches=min(PANEL_SCAN, TEST_LIMIT) if TEST_LIMIT else PANEL_SCAN,
    collect_ports={"logits"},
)
rows = getattr(datamodule.predict_ds, "rows", None)


def _logits(batch_out):
    for (_node, port), value in batch_out.items():
        if port == "logits" and value is not None:
            return value
    return None


shown = 0
for bi, batch_out in enumerate(collected):
    lg = _logits(batch_out)
    if lg is None or rows is None or bi >= len(rows):
        continue
    frame = utils.load_lentils_frame(rows[bi]["path"])  # rows carry resolved absolute paths
    if not frame["mask"].any():  # object frames only
        continue
    logits = lg[0].detach().float().cpu().numpy()  # [H, W, 2]
    e = np.exp(logits - logits.max(axis=-1, keepdims=True))
    fg_prob = (e / e.sum(axis=-1, keepdims=True))[..., 1]
    pred = logits.argmax(-1) >= 1
    utils.render_segmentation_panel(
        frame["cube"], fg_prob, pred,
        wavelengths=frame["wavelengths"], gt_mask=frame["mask"],
        title=Path(rows[bi]["path"]).name,
    )
    plt.show()
    shown += 1
    if shown >= 3:
        break

if shown == 0:
    print(f"(no object frames among the first {PANEL_SCAN} test frames)")

## Takeaways

- Numbers come from *your* trained pipeline: a 1-epoch smoke will look weak. For a real result, set
  `MAX_EPOCHS = 20` in `01_train.ipynb` and rerun, or launch the champion CLI in that notebook's
  closing section.
- `SegMetrics` covers the pixel task. The `evaluate.py` CLI additionally reports **image-level AUROC**
  (frame-has-object detection, champion ≈ 0.998) via direct tiled inference — run it for the full
  champion comparison.
- To trade a little accuracy for ~3× faster evaluation, rebuild DynUNet with `tile_overlap=0`
  (`tile_gaussian=False`); `examples/lentils/profile_pipeline.py` sweeps the tiling knobs.